[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Convolutional Networks


## What you will be able to do

Build a convolutional network for small pictures in PyTorch and in Keras, put each picture's channels
where that library expects them, and work out the shape that comes out of every layer.

## The idea

### The problem

A gray picture 12 pixels on a side is 144 numbers, and a dense layer gives each of those pixels its
own weight for every hidden unit. A line drawn two pixels further right lands on different pixels,
and so on different weights, so a dense network has to learn the same line separately in every place
it can appear. With 600 made-up pictures, each holding one mark somewhere on it, a dense network
learns its training pictures perfectly and then gets one test picture in five wrong.

What a network needs is a detector that asks whether a small patch holds a line, and asks it of every
patch in the picture.

### A formal definition

> A **convolutional layer** slides a small grid of weights, a **filter** or **kernel**, across a
> picture, and at every position computes the weighted sum of the pixels under it. The sums form a
> **feature map**, a new picture of how strongly each patch matched the filter. A layer with several
> filters makes several maps, and the next layer reads them as **channels**, as it would read the
> red, green and blue of a color picture.

### Why it works that way

The filter's weights are the same at every position, so a filter that has learned to find a vertical
line finds it anywhere. That sharing also keeps the layer small, since eight filters of 3 by 3
pixels are 72 weights and 8 biases, whatever the size of the picture.

A **max pooling** layer then keeps the largest value in each 2 by 2 block, which halves the height and
the width and keeps the strongest response from each block, so the next layer cares less about
exactly where a mark was. A second convolution looks at 3 by 3 patches of those smaller maps, which
cover a larger part of the original picture, so stacked layers see larger shapes.

The two libraries order the axes differently. PyTorch wants (batch, channels, height, width), which
it calls channels first, and Keras by default wants (batch, height, width, channels), channels last.
Each convolution and pooling layer changes the height and the width by one rule, in which the output
size is
(size + 2 × padding − kernel) ÷ stride + 1, rounded down. Each of the four Common errors below comes
from one of those two things.

### Where this shows up

The image models that `torchvision` and `keras.applications` provide, such as ResNet, are stacks of
these layers. The flattened size that feeds the last layer is where the `mat1 and mat2` message of
the **Reading Shape Errors** notebook comes back, and the **Recurrent Networks** notebook meets the
same question of which axis is which, for sequences in time.

### What this notebook covers

- the made-up pictures, and a filter written by hand
- what a convolution and a pooling layer do to a tensor's shape
- a small convolutional network in PyTorch, layer by layer, and how it compares with a dense one
- the same network in Keras, channels last
- four errors, each about an axis or a size

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

image = torch.zeros(1, 1, 5, 5)
image[0, 0, :, 2] = 1.0
kernel = torch.tensor([[[[-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0],
                         [-1.0, 2.0, -1.0]]]])
print(torch.nn.functional.conv2d(image, kernel))
```

```
tensor([[[[-3.,  6., -3.],
          [-3.,  6., -3.],
          [-3.,  6., -3.]]]])
```

The picture is 5 by 5 with a vertical line down its middle column, and the filter weighs a column by
2 and its two neighbors by −1. Centered on the line, it scores 2 × 3 = 6, and one column to either
side it scores −3, so the map it makes is a picture of where the line is. A trained network learns
weights like these instead of being given them.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and the pictures.

- `numpy` draws the pictures
- `torch` is PyTorch
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads, as in the **Why PyTorch
  and TensorFlow** notebook

`mark_images` makes 900 made-up pictures, 12 by 12 pixels of gray, from a generator started from a
fixed seed, so they are the same on every run. Each has a faint random background and one mark at a
random place: a horizontal line, a vertical line or the outline of a square. `images` holds the
pictures as 32-bit floats with shape (900, 12, 12), and `labels` holds each one's mark as 0, 1 or 2,
in the order `MARKS` names them. The pictures come in random order, so the first 600 are for
training and the last 300 for testing.

`draw` prints a picture as text, with `#` for a pixel brighter than 0.6.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


MARKS = ["horizontal", "vertical", "square"]


def mark_images(n=900, seed=2025):
    """Made-up 12 by 12 pictures, each a noisy background with one mark somewhere on it:
    a horizontal line, a vertical line or the outline of a square."""
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 3, n)
    images = rng.normal(0.0, 0.25, (n, 12, 12))
    for image, label in zip(images, labels):
        row, col = rng.integers(1, 8, 2)
        if label == 0:
            image[row + 2, col:col + 5] += 1.0
        elif label == 1:
            image[row:row + 5, col + 2] += 1.0
        else:
            image[row:row + 4, col:col + 4] += 1.0
            image[row + 1:row + 3, col + 1:col + 3] -= 1.0
    return images.astype("float32"), labels


def draw(image, threshold=0.6):
    """Print a picture as text, # where a pixel is above the threshold."""
    for row in np.asarray(image):
        print("".join("#" if value > threshold else "." for value in row))


images, labels = mark_images()
train_rows, test_rows = np.arange(600), np.arange(600, 900)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(images), "pictures")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 900 pictures


## Worked examples

### The pictures

In [2]:
print(images.shape, images.dtype, labels.dtype)
print("pictures of each mark:", np.bincount(labels))
for i in range(3):
    print(MARKS[labels[i]])
    draw(images[i])

(900, 12, 12) float32 int64
pictures of each mark: [315 313 272]
vertical
............
........#...
........#...
........#...
........#...
........#..#
............
............
.#..........
............
............
............
square
............
............
............
............
.......####.
.......#..#.
.......#..#.
.......####.
............
............
............
............
square
............
............
............
............
............
......####..
......#..#..
......#..#..
......####..
............
............
............


The three kinds of mark come in roughly equal numbers, 315, 313 and 272. The background's noise is
mostly below the threshold `draw` uses, and the occasional stray `#` is part of it.

### A filter written by hand

The first look's filter, applied to a real picture. `torch.nn.functional.conv2d` takes the picture
as (batch, channels, height, width), so `[None, None]` adds a batch of one and a single channel in
front of the 12 by 12 pixels.

In [3]:
vertical_filter = torch.tensor([[[[-1.0, 2.0, -1.0],
                                  [-1.0, 2.0, -1.0],
                                  [-1.0, 2.0, -1.0]]]])

picture = torch.from_numpy(images[0])[None, None]
response = torch.nn.functional.conv2d(picture, vertical_filter)
print(picture.shape, "->", response.shape)
draw(response[0, 0], threshold=3.0)

torch.Size([1, 1, 12, 12]) -> torch.Size([1, 1, 10, 10])
.......#..
.......#..
.......#..
.......#..
.......#..
..........
..........
..........
..........
..........


The response map is 10 by 10, two pixels smaller than the picture in each direction, because a 3 by 3
filter fits in only 10 positions across 12 pixels. Its strongest values, those above 3, sit in one
column, where the picture's vertical line is.

### What a convolutional layer does to the shape

`nn.Conv2d(1, 8, 3)` is a layer of eight 3 by 3 filters that reads one channel. Its weight holds all
eight filters at once, (filters, channels in, height, width), and it turns four pictures into four
stacks of eight maps. `padding=1` adds a border of zeros so that the maps keep the picture's size,
and `stride=2` moves the filter two pixels at a time.

In [4]:
torch.manual_seed(0)
batch = torch.from_numpy(images[:4]).unsqueeze(1)
print("pictures   ", tuple(batch.shape))
print("weight     ", tuple(torch.nn.Conv2d(1, 8, 3).weight.shape))
print("plain      ", tuple(torch.nn.Conv2d(1, 8, 3)(batch).shape))
print("padding=1  ", tuple(torch.nn.Conv2d(1, 8, 3, padding=1)(batch).shape))
print("stride=2   ", tuple(torch.nn.Conv2d(1, 8, 3, stride=2)(batch).shape))

for padding, stride in [(0, 1), (1, 1), (0, 2)]:
    print(padding, stride, (12 + 2 * padding - 3) // stride + 1)

pictures    (4, 1, 12, 12)
weight      (8, 1, 3, 3)
plain       (4, 8, 10, 10)
padding=1   (4, 8, 12, 12)
stride=2    (4, 8, 5, 5)
0 1 10
1 1 12
0 2 5


`unsqueeze(1)` puts the channel axis second, which turns (4, 12, 12) into (4, 1, 12, 12). The rule
from The idea gives each size: (12 − 3) ÷ 1 + 1 = 10 with no padding, 12 with a padding of 1, and
(12 − 3) ÷ 2 + 1 = 5.5, rounded down to 5, with a stride of 2.

### Max pooling keeps the largest value in each block

In [5]:
block = torch.tensor([[[[1.0, 3.0, 0.0, 2.0],
                        [4.0, 2.0, 1.0, 0.0],
                        [0.0, 1.0, 5.0, 6.0],
                        [2.0, 2.0, 7.0, 1.0]]]])
print(torch.nn.MaxPool2d(2)(block))

tensor([[[[4., 2.],
          [2., 7.]]]])


Each 2 by 2 corner of the 4 by 4 grid becomes one number, its largest: 4 from the top left, 2 from
the top right, 2 from the bottom left and 7 from the bottom right. A pooling layer has no weights.

### The shape after each layer of a small network

Two convolutions, each followed by `ReLU` and pooling, then `Flatten`, which lays each picture's maps
out in one row, and a `Linear` layer with one output for each kind of mark. Passing a batch through
the layers one at a time prints the shape each one hands on.

In [6]:
torch.manual_seed(0)
cnn = torch.nn.Sequential(
    torch.nn.Conv2d(1, 8, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Conv2d(8, 16, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(16 * 3 * 3, 3),
)

x = batch
for layer in cnn:
    x = layer(x)
    print(f"{type(layer).__name__:10}", tuple(x.shape))
print(sum(p.numel() for p in cnn.parameters()), "parameters")

Conv2d     (4, 8, 12, 12)
ReLU       (4, 8, 12, 12)
MaxPool2d  (4, 8, 6, 6)
Conv2d     (4, 16, 6, 6)
ReLU       (4, 16, 6, 6)
MaxPool2d  (4, 16, 3, 3)
Flatten    (4, 144)
Linear     (4, 3)
1683 parameters


Padding keeps each convolution's maps at the size it was given, and each pooling halves them, 12 to 6
and 6 to 3. The second convolution reads the first one's 8 maps as channels and makes 16, so
`Flatten` hands on 16 × 3 × 3 = 144 numbers for each picture. That is why the last layer is
`Linear(16 * 3 * 3, 3)`, and the whole network has 1,683 parameters.

### Training the network beside a dense one

`CrossEntropyLoss` takes the three outputs as logits and the mark as a class number, as the **Losses
and Optimizers** notebook explains. The loop takes the 600 training pictures in shuffled batches of
32 for 20 epochs, and prints the share of pictures each network gets right, its **accuracy**, on the
training and the test pictures every 5 epochs.

In [7]:
X_train = torch.from_numpy(images[train_rows]).unsqueeze(1)
y_train = torch.from_numpy(labels[train_rows])
X_test = torch.from_numpy(images[test_rows]).unsqueeze(1)
y_test = torch.from_numpy(labels[test_rows])


def train(model, epochs=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = torch.nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(0)
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), 32):
            rows = order[start:start + 32]
            optimizer.zero_grad()
            loss_fn(model(X_train[rows]), y_train[rows]).backward()
            optimizer.step()
        if epoch % 5 == 0:
            with torch.no_grad():
                train_accuracy = (model(X_train).argmax(dim=1) == y_train).float().mean().item()
                test_accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
            print(epoch, "training", round(train_accuracy, 3), "| test", round(test_accuracy, 3))


train(cnn)

5 training 0.998 | test 0.993
10 training 1.0 | test 0.993
15 training 1.0 | test 0.993
20 training 1.0 | test 0.993


By the fifth epoch the network gets 0.998 of its training pictures right and 0.993 of the test
pictures, 298 of 300. The training share reaches 1.0 by the tenth epoch, and the test share stays at
0.993.

In [8]:
torch.manual_seed(0)
dense = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Linear(144, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 3),
)
print(sum(p.numel() for p in dense.parameters()), "parameters")
train(dense)

4739 parameters
5 training 1.0 | test 0.793
10 training 1.0 | test 0.797
15 training 1.0 | test 0.8
20 training 1.0 | test 0.8


The dense network has nearly three times the parameters, 4,739, and gets every training picture
right by the fifth epoch, but only 0.8 of the test pictures, 240 of 300. Each of its hidden units has
a weight for every pixel, so what it learned is tied to where the marks sat in the training pictures,
and a mark in a new place is a new pattern to it.

### The same network in Keras with channels last

Keras puts the channel axis last by default, so the pictures get it with `[..., None]`, which turns
(600, 12, 12) into (600, 12, 12, 1). `Conv2D(8, 3, padding="same")` is the PyTorch layer's
equivalent, with `"same"` asking for the padding that keeps the size. Keras takes the input's channel
count from the shape the layer is built on, here the `Input`'s, where PyTorch's `Conv2d` is given it
as its first argument. `from_logits=True` tells the loss that the last layer gives logits.

In [9]:
keras.utils.set_random_seed(0)
cnn_keras = keras.Sequential([
    keras.Input((12, 12, 1)),
    keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(3),
])
for layer in cnn_keras.layers:
    print(f"{type(layer).__name__:13}", layer.output.shape)
print(cnn_keras.count_params(), "parameters | first kernel", tuple(cnn_keras.layers[0].kernel.shape))

Conv2D        (None, 12, 12, 8)
MaxPooling2D  (None, 6, 6, 8)
Conv2D        (None, 6, 6, 16)
MaxPooling2D  (None, 3, 3, 16)
Flatten       (None, 144)
Dense         (None, 3)
1683 parameters | first kernel (3, 3, 1, 8)


The sizes are PyTorch's with the channels moved last, and `None` for the batch, which Keras leaves
open, so a batch of any size can be given. The count is the same 1,683, and the first kernel is
stored as (height, width, channels in, filters), so the filter's height and width come first in
Keras and last in PyTorch's (8, 1, 3, 3).

In [10]:
cnn_keras.compile(
    optimizer=keras.optimizers.Adam(0.01),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = cnn_keras.fit(images[train_rows][..., None], labels[train_rows], epochs=20, batch_size=32, verbose=0)
test_loss, test_accuracy = cnn_keras.evaluate(images[test_rows][..., None], labels[test_rows], verbose=0)
print("accuracy during epochs 5, 10, 15 and 20", [round(a, 3) for a in history.history["accuracy"][4::5]])
print("test accuracy", round(test_accuracy, 3))

accuracy during epochs 5, 10, 15 and 20 [0.993, 1.0, 1.0, 1.0]
test accuracy 0.993


Keras's `history` holds each epoch's accuracy as the average over that epoch's batches, taken while
the weights were still changing, so the 0.993 it records for the fifth epoch mixes batches seen early
in the epoch with batches seen late. PyTorch's 0.998 was measured on all 600 pictures after the epoch
ended. On the test pictures, which `evaluate` scores after training, Keras's network gets 0.993
right, the same share as PyTorch's.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/12-convolutional-networks-solutions.ipynb).

**1.** Print the shape that `nn.Conv2d(1, 4, 5)` gives for `batch`, the first four pictures, and the size the rule from The idea predicts for a 5 by 5 filter with no padding.

In [11]:
# your code here

**2.** Count the parameters of `nn.Conv2d(8, 16, 3)` with `numel`, and write the count as a product of the filters, the channels, the filter's size and the biases.

In [12]:
# your code here

**3.** Write a filter that responds to horizontal lines, apply it to the first horizontal picture, `images[labels == 0][0]`, and `draw` its response with a threshold of 3.

In [13]:
# your code here

**4.** Train the PyTorch network with 4 and 8 filters in place of 8 and 16, from `torch.manual_seed(0)` with `train`, and print its parameter count.

In [14]:
# your code here

**5.** Build the Keras network with `padding="valid"` in both convolutions, print each layer's output shape, and say why the last `Dense` layer now has fewer weights.

In [15]:
# your code here

## Common errors

### `RuntimeError: Given groups=1, weight of size [8, 1, 3, 3], expected input[4, 12, 12, 1] to have 1 channels, but got 12 channels instead`

Four pictures given to the PyTorch network with their channel axis last, as Keras wants them.

In [16]:
cnn(torch.from_numpy(images[:4])[..., None])

RuntimeError: Given groups=1, weight of size [8, 1, 3, 3], expected input[4, 12, 12, 1] to have 1 channels, but got 12 channels instead

PyTorch reads the second axis as channels, so a (4, 12, 12, 1) tensor looks like four pictures of 12
channels, each 12 pixels by 1. The first layer's weight, [8, 1, 3, 3], says it reads one channel.
`unsqueeze(1)`, or `permute(0, 3, 1, 2)` on data that already has a channel axis last, moves it to
where PyTorch looks.

In [17]:
print(cnn(torch.from_numpy(images[:4])[..., None].permute(0, 3, 1, 2)).shape)

torch.Size([4, 3])


### `RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [12, 12]`

One picture, given as it comes from `images`.

In [18]:
cnn(torch.from_numpy(images[0]))

RuntimeError: Expected 3D (unbatched) or 4D (batched) input to conv2d, but got input of size: [12, 12]

`conv2d` takes a picture with a channel axis, (1, 12, 12), or a batch of them, (batch, 1, 12, 12), and
these 12 by 12 numbers are neither. `[None, None]` adds both axes, which makes a batch of one picture
with one channel.

In [19]:
with torch.no_grad():
    print(MARKS[cnn(torch.from_numpy(images[0])[None, None]).argmax().item()], "| label:", MARKS[labels[0]])

vertical | label: vertical


### `RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x200 and 288x3)`

A network written with one convolution and no padding, whose author worked out the flattened size as
8 maps of 6 by 6, half of 12.

In [20]:
torch.manual_seed(0)
miscounted = torch.nn.Sequential(
    torch.nn.Conv2d(1, 8, 3),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(8 * 6 * 6, 3),
)
miscounted(batch)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (4x200 and 288x3)

`mat1` is what `Flatten` produced, 4 pictures of 200 numbers each, and `mat2` the last layer's weight
turned around, 288 by 3, so the layer expects 288 numbers for each picture. With no padding the
convolution shrinks 12 pixels to 10, so pooling leaves 5 by 5, and 8 × 5 × 5 = 200. Pass a batch
through the layers one at a time, as the worked examples do, and read the size off.

In [21]:
x = batch
for layer in miscounted[:4]:
    x = layer(x)
print(tuple(x.shape))

(4, 200)


### ``ValueError: Computed output size would be negative. Received `inputs shape=(4, 1, 12, 12)`, `kernel shape=(3, 3, 12, 8)`, `dilation_rate=[1 1]`.``

The PyTorch layout given to a Keras convolution.

In [22]:
keras.utils.set_random_seed(0)
keras.layers.Conv2D(8, 3)(images[:4][:, None])

ValueError: Computed output size would be negative. Received `inputs shape=(4, 1, 12, 12)`, `kernel shape=(3, 3, 12, 8)`, `dilation_rate=[1 1]`.

Keras read the last axis as channels, so it saw four pictures 1 pixel high and 12 wide, with 12
channels, which is why the kernel it built has the shape (3, 3, 12, 8). A 3 by 3 filter does not fit
in a picture 1 pixel high. Put the channel axis last, with `[..., None]` on the pictures as they come,
or `np.transpose(x, (0, 2, 3, 1))` on data in PyTorch's layout.

In [23]:
keras.utils.set_random_seed(0)
print(keras.layers.Conv2D(8, 3)(np.transpose(images[:4][:, None], (0, 2, 3, 1))).shape)

(4, 10, 10, 8)


## Recap

- You can apply a filter to a picture and read the feature map it makes.
- You can work out the size a convolution or a pooling layer hands on, from its kernel, padding and
  stride.
- You can build and train a small convolutional network in PyTorch, channels first, and in Keras,
  channels last.
- You can read a channel error and a flattened-size error, and fix each one.

## What is next

The **Recurrent Networks** notebook turns from pictures to sequences, the hourly readings of a
weather station, and asks the same question of which axis is which, this time for time.

---

&#8592; **Previous:** [Overfitting and Regularization](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/11-overfitting-and-regularization.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Recurrent Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/13-recurrent-networks.ipynb) &#8594;
